### **BUILDING WHOLE GPT-2 MOE**

#### **IMPORTS NECESARIOS**

In [1]:
import sys
import torch
import torch.nn as nn
from pathlib import Path


#### **1. INTEGRATING MOE IN TRANFORMER BLOCK**

##### **1.1 ATTENTION**

In [2]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in:int, d_out:int, context_length:int, attr:dict):

        # Initialiaze the parent
        super().__init__() 

        # Get att specific params
        num_heads:int = attr.get('num_heads', 6)
        dropout:float = attr.get('dropout', 0.1)
        qkv_bias:bool = attr.get('qkv_bias', False)

        # Get basic parameters
        self.d_out = d_out
        self.num_heads = num_heads
        # Force sum(dim(heads)) =  dout
        self.head_dim = d_out // num_heads 

        # Define the QKV matrices -> dk = dv = dout
        self.W_query = nn.Linear(d_in, d_out, bias = qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias = qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias = qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)  
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length),
                       diagonal = 1)   
        )

    def forward(self, x):
        # Get the shape (B, N, d_in)
        b, num_tokens, d_in = x.shape
        # Define the KQV matrices (B, num_tokens, d_out)
        keys = self.W_key(x) # Shape: (b, num_tokens, d_out)
        queries = self.W_query(x)
        values = self.W_value(x)

        # We modify shapes to consider all the heads
        # (b, num_tokens, d_out) -> (b, num_tokens, num_heads, head_dim)
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim) 
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)

        # Transpose: (b, num_tokens, num_heads, head_dim) -> (b, num_heads, num_tokens, head_dim)
        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        # Compute scaled dot-product attention 
        attn_scores = queries @ keys.transpose(2, 3)  

        # Original mask truncated to the number of tokens and converted to boolean
        # Addapted to the real amount of tokens in case input is not padded to obtain fixed N (context_length)
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]

        # Use the mask to fill attention scores -> upper diagonal MLM
        attn_scores.masked_fill_(mask_bool, -torch.inf)
        # Get attention weights
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        # Shape: (b, num_heads, num_tokens, head_dim) -> (b, num_tokens, num_heads, head_dim)
        context_vec = (attn_weights @ values).transpose(1, 2) 
        
        # Combine heads, where d_out -> num_heads * head_dim
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
        context_vec = self.out_proj(context_vec) # optional projection

        return context_vec

##### **1.2 NORMALIZATION**

In [3]:
class LayerNorm(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        norm_x = (x - mean) / torch.sqrt(var + self.eps)
        return self.scale * norm_x + self.shift

##### **1.3 MOE BLOCK**

In [4]:
src_path = str(Path("/home/carlo/cuarto/GPT2_MOE/src").resolve())

# Asegurar que src tiene prioridad
if src_path in sys.path:
    sys.path.remove(src_path)
sys.path.insert(0, src_path)

from gpt2_moe.model.moe import MOE

##### **1.4 TRANSFORMER BLOCK**

In [5]:
# Relative imports 
class TransformerBlock(nn.Module):
    def __init__(self, attr:dict):
        # Initialize parent
        super().__init__()
        # Common params
        self.d_in = attr["emb_dim"]
        self.d_out = attr["emb_dim"]
        self.cl = attr["context_length"]
        # Block specific params
        self.att_related = attr["attention_related"]
        self.moe_related = attr["moe_related"]
        
        # Define the attention module -> ok
        self.att = MultiHeadAttention(
           self.d_in,
           self.d_out,
           self.cl,
           self.att_related)

        # Define the ff layer
        self.ff = MOE(
                    self.d_in, 
                    self.d_out, 
                    self.moe_related) 

        # Define normalizations (within and previous)
        self.norm1 = LayerNorm(self.d_in)
        self.norm2 = LayerNorm(self.d_in)
        
        # Define the dropout
        self.drop_shortcut = nn.Dropout(attr['dropout'])

    def forward(self, x:torch.Tensor)->torch.Tensor:
        # Shortcut connection for attention block
        shortcut = x
        x = self.norm1(x)
        # (B, N, de) ->(B, N, de)
        x = self.att(x)  
        print(f'(TFBLOCK) Aspect of attention output ({x.shape})')
        x = self.drop_shortcut(x)
        # Add the original input back
        x = x + shortcut  

        # Shortcut connection for feed forward block
        shortcut = x
        x = self.norm2(x)
        # Go through ff Layer
        x = self.ff(x)
        print(f'(TFBLOCK) Aspect of FF output ({x.shape})')
        x = self.drop_shortcut(x)
        # Add the original input back
        x = x + shortcut 

        print(f'(TFBLOCK) Aspect of block output ({x.shape})')
 

        return x

In [6]:
# Define config for the block
block_config = {
    "context_length": 1024, 
    "emb_dim": 768, 
    "dropout":0.1,     
    "attention_related":{
        "num_heads":8,
        "dropout":0.1,
        "qkv_bias":False
    },
    "moe_related":{
        "experts":3,
        "k_routing":1,
        "hidden_dim":768,
    }
}

In [7]:
# Define mock tensor -> currently embedded
mock = torch.randn(size = (10, 100, 768))

# Get the block
tf_block = TransformerBlock(block_config)

# Test the function
result = tf_block(mock)



(TFBLOCK) Aspect of attention output (torch.Size([10, 100, 768]))
(MOE) CALLING GATING MECHANISM
(router) Output of the linear:torch.Size([1000, 3])
(router) Output of the top_k:torch.Size([1000, 1])--(torch.Size([1000, 1]))
(router) Aspect of the -inf mask:torch.Size([1000, 3])
(router) BUILDING INDEXING MECHANISM
(router) ·Aspect of the first index:torch.Size([1000, 1])
(router) ·Aspect of the second index:torch.Size([1000, 1])
(router) ·Aspect of the later inf maskk:torch.Size([1000, 3])
(router) Aspect of the normalized weights:torch.Size([1000, 1])
(MOE) Aspect of the container for output tensor: torch.Size([1000, 768])
(MOE) EXPERT (0)
(expert) Output of the first linear:torch.Size([337, 768])
(expert) Output of the second linear:torch.Size([337, 768])
(expert) Aspect of the expert weights: torch.Size([337, 1])
(MOE) EXPERT (1)
(expert) Output of the first linear:torch.Size([340, 768])
(expert) Output of the second linear:torch.Size([340, 768])
(expert) Aspect of the expert weigh

#### **2. INTEGRATING MOE IN GPT BLOCK**


##### **2.1 GPT BLOCK**

##### **2.2 PARAM DEFINITION**